# 2.configurable_fields替换提示词

知识点讲解：configurable_fields 运行时替换提示词模板

1. 提示词模板也是 Runnable，同样支持运行时配置
   - configurable_fields 不是 LLM 专属能力，任何 Runnable 都可使用
   - PromptTemplate 的 template 字段可被声明为运行时可配置
   - 这使得"同一条链、不同提示词策略"成为可能，无需重建链

2. 运行时替换提示词的典型场景
   - A/B 测试：对比不同提示词写法的效果，链结构保持不变
   - 多任务复用：同一变量（如 subject）配合不同指令（冷笑话/藏头诗/摘要）
   - 多语言支持：根据用户语言偏好切换中英文提示词
   - 提示词热更新：从数据库读取最新模板，无需重启服务

3. 可配置字段名必须是组件的真实属性名（本文件的已知 Bug）
   - PromptTemplate 的模板字符串字段名为 template（不是 template_demo）
   - 若传入不存在的字段名，configurable_fields 会立即抛出 ValueError
   - 可通过 print(prompt.model_fields.keys()) 查看全部可配置的字段名
   - 【本文件 Bug】：原代码使用 template_demo=ConfigurableField(...) 是错误的
   - 正确写法应为 template=ConfigurableField(...)
   - 该 Bug 会在第 77-79 行立即抛出：ValueError: Configuration key template_demo not found
   - 保留该错误写法是为了演示字段名错误的后果，正确写法见文末注释

4. 替换 template 时的变量兼容性要求
   - 新模板中的变量应与原模板保持一致（本例都使用 {subject}）
   - 若新模板引入了原模板没有的变量，invoke 时会因缺少入参而报错
   - 若新模板缺少原模板的变量，多余入参通常被忽略（取决于版本行为）

5. configurable_fields 的三种典型应用对象
   - PromptTemplate.template：替换提示词文本（本文件）
   - ChatOpenAI.temperature / model_name：调整模型参数（见本章第 1 节）
   - Retriever.search_kwargs：动态调整检索的 Top-K 数量

6. 与 configurable_alternatives 的区别
   - configurable_fields：修改组件内部的某个字段值（细粒度改参）
   - configurable_alternatives：整体替换为另一个预注册的组件（粗粒度换件）
   - 替换提示词文本用前者，切换到完全不同类型的 prompt 用后者

7. 数据流转过程（假设使用正确的字段名 template）
   - prompt.invoke({"subject": "程序员"}, config={"configurable": {"prompt_template": 新模板}})
   - → 框架读取 configurable，用新模板字符串重建 PromptTemplate 副本
   - → 副本填充 subject 变量 → StringPromptValue
   - → to_string() → "请写一篇关于程序员主题的藏头诗"


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import ConfigurableField
prompt = PromptTemplate.from_template("请写一篇关于{subject}主题的冷笑话").configurable_fields(
    template_demo=ConfigurableField(id="prompt_template"),
)
content = prompt.invoke(
    {"subject": "程序员"},
    config={"configurable": {"prompt_template": "请写一篇关于{subject}主题的藏头诗"}}
).to_string()
print(content)